In [1]:
import pandas as pd
import os
import plotly.express as px


from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Connexion NeonDB

In [2]:
load_dotenv("secrets.env")

True

Récupération des paramètres NEON

In [3]:
NEON_USER=os.getenv("NEON_USER")
NEON_PASSWORD=os.getenv("NEON_PASSWORD")
NEON_HOST=os.getenv("NEON_HOST")
NEON_DBNAME=os.getenv("NEON_DBNAME")

engine = create_engine(
    f"postgresql+psycopg2://{NEON_USER}:{NEON_PASSWORD}@{NEON_HOST}/{NEON_DBNAME}?sslmode=require"
)

## Requête essai

In [4]:
with engine.connect() as conn:
    df_villes = pd.read_sql("SELECT * FROM villes", engine)
    df_meteos = pd.read_sql("SELECT * from meteos", engine)
    df_hotels = pd.read_sql("SELECT * from hotels", engine)
    df_condition = pd.read_sql("SELECT * FROM condition", engine)
    

In [5]:
display(df_villes.head(2))
display(df_hotels.head(2))
display(df_meteos.head(2))
display(df_condition.head(2))


,ville_id,nom,latitude,longitude,latitude2,longitude2
0,1,Le Mont Saint Michel,48.635523,-1.510257,48.635523,-1.510257
1,2,Saint Malo,48.649518,-2.026041,48.649518,-2.026041


,ville,name,description,url,rating,latitude,longitude,prix,ville_id
0,Aigues-Mortes,Hotel Lou Marquès,Le Lou Marquès est situé aux Saintes-Maries-de...,https://www.booking.com/hotel/fr/lou-marques.f...,"Avec une note de 9,1",43.455125,4.428740,€ 708,26
1,Aigues-Mortes,MY Hotel Residences - A106,L’établissement MY Hotel Residences - A106 vou...,https://www.booking.com/hotel/fr/my-residences...,"Avec une note de 9,0",43.559329,4.069145,€ 859,26


,ville_id,date,condition,temperature,duree_ensoleillement,pluie,vent
0,1,2025-11-28,6,5.489741,0.000,6.150000,46.332436
1,1,2025-11-29,6,6.329223,24423.959,11.400001,40.417915


,description,code
0,Soleil,1
1,Nuageux,2


# Top 5 des villes pour la semaine du 14 au 21 Novembre

In [6]:
query = """
SELECT 
    v.nom,
    v.latitude,
    v.longitude,
    m.temperature,
    m.duree_soleil,
    m.vent,
    m.pluie,
	c.code,
    c.description temps,
    v.ville_id
FROM villes v
	INNER JOIN 
		(
			SELECT 
				ville_id, 
				avg(temperature) temperature, 
				avg(duree_ensoleillement) duree_soleil, 
				avg(vent) vent,
				avg(pluie)pluie, 
				round(avg(condition),0) condition
			FROM meteos
			GROUP BY
				ville_id
		) m on v.ville_id=m.ville_id
	INNER JOIN condition c ON m.condition=c.code
ORDER BY 
    c.code,
	m.duree_soleil DESC,
	m.temperature DESC,
	m.pluie,
	m.vent
;
"""

with engine.connect() as conn:
    result = conn.execute(text(query))
    top5_destinations_sql = pd.DataFrame(result.fetchall(), columns=result.keys())

top5 = top5_destinations_sql.head(5).copy()
top5["rang"] = range(1, len(top5) + 1)
top5

,nom,latitude,longitude,temperature,duree_soleil,vent,pluie,code,temps,ville_id,rang
0,Collioure,42.525050,3.083155,12.521052,22315.975050,16.681920,2.265625,2,Nuageux,28,1
1,Bormes-les-Mimosas,43.150697,6.341928,9.699339,20700.723941,20.979544,2.025000,2,Nuageux,19,2
2,Strasbourg,48.584614,7.750713,5.721698,20114.596600,16.722315,1.193750,2,Nuageux,9,3
3,Eguisheim,48.044797,7.307962,5.435027,19595.628000,14.692560,0.862500,2,Nuageux,12,4
4,Colmar,48.077752,7.357964,6.062992,19557.617875,13.640510,0.893750,2,Nuageux,11,5


In [ ]:
top5["rang_str"] = top5["rang"].astype(str)

custom_colors = {
    "1": "blue",
    "2": "red",
    "3": "green",
    "4": "orange",
    "5": "purple"
}


fig = px.scatter_mapbox(
    top5,
    lat="latitude",
    lon="longitude",
    size="temperature",
    color="rang_str",  
    color_discrete_map=custom_colors,
    hover_name="nom",
    hover_data={
        "temperature": True,
        "duree_soleil": True,
        "vent": True,
        "pluie": True,
        "temps": True,
        "rang": False,
        "latitude": False,
        "longitude": False
    },
    mapbox_style="open-street-map",
    zoom=4,
    title="Classement des destinations selon la météo"
)

fig.update_layout(
    legend_title_text="Classement météo",
    width=550,   
    height=600,  
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.25,
        xanchor="center",
        x=0.5,
        font=dict(size=13, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1
    )
)

fig.show()


## 20 hotels et où ils sont

In [8]:
top5

,nom,latitude,longitude,temperature,duree_soleil,vent,pluie,code,temps,ville_id,rang,rang_str
0,Collioure,42.525050,3.083155,12.521052,22315.975050,16.681920,2.265625,2,Nuageux,28,1,1
1,Bormes-les-Mimosas,43.150697,6.341928,9.699339,20700.723941,20.979544,2.025000,2,Nuageux,19,2,2
2,Strasbourg,48.584614,7.750713,5.721698,20114.596600,16.722315,1.193750,2,Nuageux,9,3,3
3,Eguisheim,48.044797,7.307962,5.435027,19595.628000,14.692560,0.862500,2,Nuageux,12,4,4
4,Colmar,48.077752,7.357964,6.062992,19557.617875,13.640510,0.893750,2,Nuageux,11,5,5


In [ ]:
top5_villes = top5['ville_id'].tolist()

for ville in top5_villes:
    query = f"""
        SELECT 
            h.name AS hotel,
            v.nom,
            h.latitude,
            h.longitude,
            CAST(REPLACE(SUBSTR(h.rating, 18), ',', '.') AS FLOAT) AS note,
            prix
        FROM villes v
        INNER JOIN hotels h ON h.ville_id = v.ville_id
        WHERE v.ville_id = {ville}
        ORDER BY note DESC
    """

    with engine.connect() as conn:
        result = conn.execute(text(query))
        top_hotels = pd.DataFrame(result.fetchall(), columns=result.keys())

    if top_hotels.empty:
        print(f"⚠️ Pas de données pour ville_id={ville}")
        continue

    top_hotels = top_hotels.dropna(subset=["note"])

    fig = px.scatter_map(
        top_hotels,
        lat="latitude",
        width=500,
        height=500,
        lon="longitude",
        hover_name="hotel",
        hover_data={"nom": True, "note": True, "prix":True, "latitude": False, "longitude": False},
        color="note",
        color_continuous_scale="Viridis",
        range_color=[7, 10], 
        size="note", 
        size_max=20, 
        zoom=9,      
        title=f"🏨 Hôtels à {top_hotels['nom'].iloc[0]}"
    )

    fig.update_layout(
        coloraxis_colorbar=dict(title="Note"),
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.show()
